### Importación de librerías.

In [1]:
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql import functions as f
from pyspark.sql.types import NumericType, StringType
import os
from pyspark import StorageLevel

### Conexión

In [2]:
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += os.pathsep + r"C:\hadoop\bin"

In [3]:
spark = (SparkSession.builder.appName("Proyecto_PySpark").master("local[*]").getOrCreate())

/usr/local/lib/python3.11/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/25 23:51:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Lectura de archivos

In [4]:
df_2025_1_pd = pd.read_excel('../working_dir/2025T1.xlsx')
df_2025_2_pd = pd.read_excel('../working_dir/2025T2.xlsx')
df_2025_3_pd = pd.read_excel('../working_dir/2025T3.xlsx')
df_2025_4_pd = pd.read_excel('../working_dir/2025T4.xlsx')
df_2026_1_pd = pd.read_excel('../working_dir/2026T1.xlsx')

print('df_2025T1 tiene', df_2025_1_pd.shape[0], 'registros')
print('df_2025T2 tiene', df_2025_2_pd.shape[0], 'registros')
print('df_2025T3 tiene', df_2025_3_pd.shape[0], 'registros')
print('df_2025T4 tiene', df_2025_4_pd.shape[0], 'registros')
print('df_2026T1 tiene', df_2026_1_pd.shape[0], 'registros')

df_2025T1 tiene 51588 registros
df_2025T2 tiene 51167 registros
df_2025T3 tiene 51583 registros
df_2025T4 tiene 49338 registros
df_2026T1 tiene 49843 registros


### Dejando las columnas útiles.

In [5]:
columnas_conservar = [
    'P05D01',
    'P02A03',
    'P05C07A',
    'P05C07B',
    'P05H01A',
    'P03A03A',
    'P05C16',
    'DOMINIO',
    'OCUPADOS',
    'NUM_HOGAR',
    'NUM_PERSONA',
    'FACTOR',
    'ANIO',
    'TRIMESTRE'
]

df_2025_1_pd = df_2025_1_pd[columnas_conservar]
df_2025_2_pd = df_2025_2_pd[columnas_conservar]
df_2025_3_pd = df_2025_3_pd[columnas_conservar]
df_2025_4_pd = df_2025_4_pd[columnas_conservar]
df_2026_1_pd = df_2026_1_pd[columnas_conservar]

### Conviertiendo a DF de spark.

In [6]:
df_2025_1 = spark.createDataFrame(df_2025_1_pd)
df_2025_2 = spark.createDataFrame(df_2025_1_pd)
df_2025_3 = spark.createDataFrame(df_2025_1_pd)
df_2025_4 = spark.createDataFrame(df_2025_1_pd)
df_2026_1 = spark.createDataFrame(df_2025_1_pd)

del df_2025_1_pd, df_2025_2_pd, df_2025_3_pd, df_2025_4_pd, df_2026_1_pd

### Creando columnas nuevas

In [7]:
df_2025_1 = df_2025_1.withColumns({'periodo_archivo': f.lit('2025T1'),
                                  'anio_archivo': f.lit(2025),
                                  'trimestre_calendario': f.lit(1)})
df_2025_2 = df_2025_2.withColumns({'periodo_archivo': f.lit('2025T2'),
                                  'anio_archivo': f.lit(2025),
                                  'trimestre_calendario': f.lit(2)})
df_2025_3 = df_2025_3.withColumns({'periodo_archivo': f.lit('2025T3'),
                                  'anio_archivo': f.lit(2025),
                                  'trimestre_calendario': f.lit(3)})
df_2025_4 = df_2025_4.withColumns({'periodo_archivo': f.lit('2025T4'),
                                  'anio_archivo': f.lit(2025),
                                  'trimestre_calendario': f.lit(4)})
df_2026_1 = df_2026_1.withColumns({'periodo_archivo': f.lit('2026T1'),
                                  'anio_archivo': f.lit(2026),
                                  'trimestre_calendario': f.lit(1)})

### Función de faltantes

In [8]:
def tabla_faltantes(df):

    # Cantidad total de registros
    total_registros = df.count()

    expresiones = []

    for columna in df.columns:

        tipo = df.schema[columna].dataType

        # Columnas numéricas
        if isinstance(tipo, NumericType):
            condicion_faltante = (f.col(columna).isNull() | f.isnan(f.col(columna)))

        # Columnas de texto
        elif isinstance(tipo, StringType):
            condicion_faltante = (
                f.col(columna).isNull() | (f.trim(f.col(columna)) == ""))

        # Otros tipos
        else:
            condicion_faltante = f.col(columna).isNull()

        # Expresión para contar faltantes
        expresiones.append(f.sum(f.when(condicion_faltante, 1).otherwise(0)).alias(columna))

    # Spark calcula todos los faltantes
    faltantes = df.agg(*expresiones).first().asDict()

    # Crear DataFrame de pandas
    df_resultado = pd.DataFrame({"variable": faltantes.keys(), "cantidad_faltantes": faltantes.values()})

    # Calcular porcentaje
    df_resultado["porcentaje_faltantes"] = (df_resultado["cantidad_faltantes"] / total_registros * 100).round(2)

    return df_resultado

### Faltantes antes del primer filtro

In [9]:
tabla_faltantes(df_2025_1)

,variable,cantidad_faltantes,porcentaje_faltantes
0,P05D01,38169,73.99
1,P02A03,0,0.00
2,P05C07A,29315,56.83
3,P05C07B,29315,56.83
4,P05H01A,29315,56.83
5,P03A03A,6928,13.43
6,P05C16,29315,56.83
7,DOMINIO,0,0.00
8,OCUPADOS,29315,56.83
9,NUM_HOGAR,0,0.00


In [10]:
tabla_faltantes(df_2025_2)

,variable,cantidad_faltantes,porcentaje_faltantes
0,P05D01,38169,73.99
1,P02A03,0,0.00
2,P05C07A,29315,56.83
3,P05C07B,29315,56.83
4,P05H01A,29315,56.83
5,P03A03A,6928,13.43
6,P05C16,29315,56.83
7,DOMINIO,0,0.00
8,OCUPADOS,29315,56.83
9,NUM_HOGAR,0,0.00


In [11]:
tabla_faltantes(df_2025_3)

,variable,cantidad_faltantes,porcentaje_faltantes
0,P05D01,38169,73.99
1,P02A03,0,0.00
2,P05C07A,29315,56.83
3,P05C07B,29315,56.83
4,P05H01A,29315,56.83
5,P03A03A,6928,13.43
6,P05C16,29315,56.83
7,DOMINIO,0,0.00
8,OCUPADOS,29315,56.83
9,NUM_HOGAR,0,0.00


In [12]:
tabla_faltantes(df_2025_4)

,variable,cantidad_faltantes,porcentaje_faltantes
0,P05D01,38169,73.99
1,P02A03,0,0.00
2,P05C07A,29315,56.83
3,P05C07B,29315,56.83
4,P05H01A,29315,56.83
5,P03A03A,6928,13.43
6,P05C16,29315,56.83
7,DOMINIO,0,0.00
8,OCUPADOS,29315,56.83
9,NUM_HOGAR,0,0.00


In [13]:
tabla_faltantes(df_2026_1)

,variable,cantidad_faltantes,porcentaje_faltantes
0,P05D01,38169,73.99
1,P02A03,0,0.00
2,P05C07A,29315,56.83
3,P05C07B,29315,56.83
4,P05H01A,29315,56.83
5,P03A03A,6928,13.43
6,P05C16,29315,56.83
7,DOMINIO,0,0.00
8,OCUPADOS,29315,56.83
9,NUM_HOGAR,0,0.00


### Primer filtro

In [14]:
count_i_1 = df_2025_1.count()
count_i_2 = df_2025_2.count()
count_i_3 = df_2025_3.count()
count_i_4 = df_2025_4.count()
count_i_5 = df_2026_1.count()

valor = f.col("P05D01").cast("double")
df_2025_1 = df_2025_1.filter((f.col('P02A03') >= 15) &
                                      (f.col('OCUPADOS') == 1) &
                                      (f.col('P05C16') >= 1)& 
                                      (f.col('P05C16') <= 4) &
                                      (valor.isNotNull()) &
                                      (~f.isnan(valor)) &
                                      (valor > 0) &
                                      (valor < float("inf")))

df_2025_2 = df_2025_2.filter((f.col('P02A03') >= 15) &
                                      (f.col('OCUPADOS') == 1) &
                                      (f.col('P05C16') >= 1)& 
                                      (f.col('P05C16') <= 4) &
                                      (valor.isNotNull()) &
                                      (~f.isnan(valor)) &
                                      (valor > 0) &
                                      (valor < float("inf")))

df_2025_3 = df_2025_3.filter((f.col('P02A03') >= 15) &
                                      (f.col('OCUPADOS') == 1) &
                                      (f.col('P05C16') >= 1)& 
                                      (f.col('P05C16') <= 4) &
                                      (valor.isNotNull()) &
                                      (~f.isnan(valor)) &
                                      (valor > 0) &
                                      (valor < float("inf")))

df_2025_4 = df_2025_4.filter((f.col('P02A03') >= 15) &
                                      (f.col('OCUPADOS') == 1) &
                                      (f.col('P05C16') >= 1)& 
                                      (f.col('P05C16') <= 4) &
                                      (valor.isNotNull()) &
                                      (~f.isnan(valor)) &
                                      (valor > 0) &
                                      (valor < float("inf")))

df_2026_1 = df_2026_1.filter((f.col('P02A03') >= 15) &
                                      (f.col('OCUPADOS') == 1) &
                                      (f.col('P05C16') >= 1)& 
                                      (f.col('P05C16') <= 4) &
                                      (valor.isNotNull()) &
                                      (~f.isnan(valor)) &
                                      (valor > 0) &
                                      (valor < float("inf")))


count_f_1 = df_2025_1.count()
count_f_2 = df_2025_2.count()
count_f_3 = df_2025_3.count()
count_f_4 = df_2025_4.count()
count_f_5 = df_2026_1.count()

print('Se afectaron', count_i_1 - count_f_1, 'filas en df_2025_1')
print('Se afectaron', count_i_2 - count_f_2, 'filas en df_2025_2')
print('Se afectaron', count_i_3 - count_f_3, 'filas en df_2025_3')
print('Se afectaron', count_i_4 - count_f_4, 'filas en df_2025_4')
print('Se afectaron', count_i_5 - count_f_5, 'filas en df_2026_1')

Se afectaron 38169 filas en df_2025_1
Se afectaron 38169 filas en df_2025_2
Se afectaron 38169 filas en df_2025_3
Se afectaron 38169 filas en df_2025_4
Se afectaron 38169 filas en df_2026_1


### Cambiando el código

In [15]:
def normalizar_codigos(df, columnas):
    
    for columna in columnas:
        df = df.withColumn(columna, f.regexp_replace(f.trim(f.col(columna).cast("double")), r"\.0+$", ""))
    
    return df

columnas_codigos = ["P03A03A", "P05C16", "DOMINIO", "OCUPADOS", "TRIMESTRE"]

df_2025_1 = normalizar_codigos(df_2025_1, columnas_codigos)
df_2025_2 = normalizar_codigos(df_2025_2, columnas_codigos)
df_2025_3 = normalizar_codigos(df_2025_3, columnas_codigos)
df_2025_4 = normalizar_codigos(df_2025_4, columnas_codigos)
df_2026_1 = normalizar_codigos(df_2026_1, columnas_codigos)

### Faltantes antes de la unión.

In [16]:
tabla_faltantes(df_2025_1)

,variable,cantidad_faltantes,porcentaje_faltantes
0,P05D01,0,0.0
1,P02A03,0,0.0
2,P05C07A,0,0.0
3,P05C07B,0,0.0
4,P05H01A,0,0.0
5,P03A03A,0,0.0
6,P05C16,0,0.0
7,DOMINIO,0,0.0
8,OCUPADOS,0,0.0
9,NUM_HOGAR,0,0.0


In [17]:
tabla_faltantes(df_2025_2)

,variable,cantidad_faltantes,porcentaje_faltantes
0,P05D01,0,0.0
1,P02A03,0,0.0
2,P05C07A,0,0.0
3,P05C07B,0,0.0
4,P05H01A,0,0.0
5,P03A03A,0,0.0
6,P05C16,0,0.0
7,DOMINIO,0,0.0
8,OCUPADOS,0,0.0
9,NUM_HOGAR,0,0.0


In [18]:
tabla_faltantes(df_2025_3)

,variable,cantidad_faltantes,porcentaje_faltantes
0,P05D01,0,0.0
1,P02A03,0,0.0
2,P05C07A,0,0.0
3,P05C07B,0,0.0
4,P05H01A,0,0.0
5,P03A03A,0,0.0
6,P05C16,0,0.0
7,DOMINIO,0,0.0
8,OCUPADOS,0,0.0
9,NUM_HOGAR,0,0.0


In [19]:
tabla_faltantes(df_2025_4)

,variable,cantidad_faltantes,porcentaje_faltantes
0,P05D01,0,0.0
1,P02A03,0,0.0
2,P05C07A,0,0.0
3,P05C07B,0,0.0
4,P05H01A,0,0.0
5,P03A03A,0,0.0
6,P05C16,0,0.0
7,DOMINIO,0,0.0
8,OCUPADOS,0,0.0
9,NUM_HOGAR,0,0.0


In [20]:
tabla_faltantes(df_2026_1)

,variable,cantidad_faltantes,porcentaje_faltantes
0,P05D01,0,0.0
1,P02A03,0,0.0
2,P05C07A,0,0.0
3,P05C07B,0,0.0
4,P05H01A,0,0.0
5,P03A03A,0,0.0
6,P05C16,0,0.0
7,DOMINIO,0,0.0
8,OCUPADOS,0,0.0
9,NUM_HOGAR,0,0.0


Ahora ninguna columna tiene faltantes.

### Uniendo los DataFrame.

In [21]:
df_2025 = (df_2025_1
    .unionByName(df_2025_2)
    .unionByName(df_2025_3)
    .unionByName(df_2025_4)
)

nuevos_nombres = {
    "P05D01": "salario_mensual",
    "P02A03": "edad",
    "P05C07A": "antiguedad_anios",
    "P05C07B": "antiguedad_meses",
    "P05H01A": "horas_semanales",
    "P03A03A": "nivel_educativo",
    "P05C16": "categoria_ocupacional",
    "DOMINIO": "dominio",
    "OCUPADOS": "ocupado"
}

for columna_original, columna_nueva in nuevos_nombres.items():
    df_2025 = df_2025.withColumnRenamed(columna_original, columna_nueva)
    df_2026_1 = df_2026_1.withColumnRenamed(columna_original, columna_nueva)

### Columna antiguedad

In [22]:
df_2025 = df_2025.withColumn("antiguedad_total", f.col("antiguedad_anios") + f.col("antiguedad_meses") / 12)
df_2026_1 = df_2026_1.withColumn("antiguedad_total", f.col("antiguedad_anios") + f.col("antiguedad_meses") / 12)

### Segundo filtro

In [23]:
# Guardamos en memoria/disco los DataFrames ya preparados
df_2025 = df_2025.persist(StorageLevel.MEMORY_AND_DISK)
df_2026_1 = df_2026_1.persist(StorageLevel.MEMORY_AND_DISK)


# Columnas convertidas a double
edad = f.col("edad").cast("double")
antiguedad_anios = f.col("antiguedad_anios").cast("double")
antiguedad_meses = f.col("antiguedad_meses").cast("double")
antiguedad_total = f.col("antiguedad_total").cast("double")
horas_semanales = f.col("horas_semanales").cast("double")


# Una sola condición
condicion = (

    # Edad finita y >= 15
    (edad < float("inf")) &

    # Antigüedad >= 0
    (antiguedad_anios >= 0) &

    # Meses entre 0 y 11 y enteros
    (antiguedad_meses >= 0) &
    (antiguedad_meses <= 11) &
    (antiguedad_meses == f.floor(antiguedad_meses)) &

    # Antigüedad total <= edad
    (antiguedad_total <= edad) &

    # Horas válidas
    (horas_semanales > 0) &
    (horas_semanales <= 168)
)

### Filtrado y conteo.

In [24]:
def filtrar_y_contar(df, condicion):

    resultado = df.agg(
        
        f.count("*").alias("antes"),
        
        f.sum(
            f.when(condicion, 1).otherwise(0)
        ).alias("despues")
        
    ).first()

    antes = resultado["antes"]
    despues = resultado["despues"]
    eliminados = antes - despues

    # Aplicamos el filtro
    df_filtrado = df.filter(condicion)

    return df_filtrado, antes, despues, eliminados

In [25]:
df_2025, count_i_1, count_f_1, eliminados_2025 = filtrar_y_contar(
    df_2025,
    condicion
)

df_2026_1, count_i_2, count_f_2, eliminados_2026 = filtrar_y_contar(
    df_2026_1,
    condicion
)

print("2025")
print("Registros antes:", count_i_1)
print("Registros después:", count_f_1)
print("Registros eliminados:", eliminados_2025)

print("\n2026")
print("Registros antes:", count_i_2)
print("Registros después:", count_f_2)
print("Registros eliminados:", eliminados_2026)

2025
Registros antes: 53676
Registros después: 53676
Registros eliminados: 0

2026
Registros antes: 13419
Registros después: 13419
Registros eliminados: 0


### Duplicado

In [27]:
duplicados_2025 = (df_2025.groupBy("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA").count().filter(f.col("count") > 1))
duplicados_2025.show()

[Stage 98:=====================================>                (88 + 32) / 128]

+---------------+---------+-----------+-----+
|periodo_archivo|NUM_HOGAR|NUM_PERSONA|count|
+---------------+---------+-----------+-----+
+---------------+---------+-----------+-----+



In [28]:
duplicados_2026_1 = (df_2026_1.groupBy("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA").count().filter(f.col("count") > 1))
duplicados_2026_1.show()

+---------------+---------+-----------+-----+
|periodo_archivo|NUM_HOGAR|NUM_PERSONA|count|
+---------------+---------+-----------+-----+
+---------------+---------+-----------+-----+



No hay duplicados.

### Preguntas

- ¿Por qué IV de 2025 no puede apilarse por posición de columnas con los otros archivos?
Porque tiene más columnas que los otros archivos, 302, mientras que los otros tienen 270.

- ¿Qué diferencia existe entre un dato ausente porque la pregunta no corresponde y una respuesta no registrada?
Un dato ausente de pregunta no contestada no es información no percibida, mientras que el de una respuesta no registrada si tiene información no percibida.

- ¿Por qué una persona observada en dos períodos no debe eliminarse como duplicado del conjunto longitudinal?
Porque una persona puede aparecer varias veces, ya que cada registro es una observación en un período distinto de tiempo.

- ¿Por qué el número de registros de la base filtrada no representa a todos los trabajadores del país?
Porque al aplicar filtros estamos sesgando la muestra a ciertas características.

### Exportando a parquet

In [26]:
df_2025.write \
    .mode("overwrite") \
    .parquet("df_2025.parquet")

df_2026_1.write \
    .mode("overwrite") \
    .parquet("df_2026.parquet")

26/09/25 03:52:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/25 03:52:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/25 03:52:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/25 03:52:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/25 03:52:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/09/25 03:52:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/09/25 03:52:20 WARN MemoryManager: Total allocation exceeds 95.